In [5]:
from backtesting import Backtest, Strategy
from backtesting.lib import crossover
from backtesting.test import GOOG
import talib

In [6]:
def calc_tp_sl(last_close: float, last_atr: float, long: bool,
    rr_ratio: int = 10
) -> tuple[float, float]:
    sl: float = 0.0
    tp: float = 0.0
    diff: float = last_atr
    if long:
        sl = last_close - diff
        tp = last_close + (rr_ratio * diff)
    else:
        sl = last_close + diff
        tp = last_close - (rr_ratio * diff)

    return (tp, sl)

In [7]:
class TrendFollow(Strategy):
    def init(self):
        close = self.data.Close
        high = self.data.High
        low = self.data.Low

        self.fast_ma = self.I(talib.EMA, close, 10)
        self.slow_ma = self.I(talib.EMA, close, 30)
        self.atr = self.I(
            talib.ATR, high=high, close=close, low=low, timeperiod=14,
            overlay=False
        )
        self.rsi = self.I(talib.RSI, close, 14, overlay=False)

    def next(self):
        last_close = self.data.Close[-1]
        last_atr = self.atr[-1]

        if crossover(self.fast_ma, self.slow_ma): # type: ignore
        # if self.rsi[-1] < 40: # type: ignore
            tp, sl = calc_tp_sl(last_close, last_atr, long=True)
            self.buy(tp=tp, sl=sl)
        elif crossover(self.slow_ma, self.fast_ma): # type: ignore
        # elif self.rsi[-1] > 60:
            tp, sl = calc_tp_sl(last_close, last_atr, long=False)
            self.sell(tp=tp, sl=sl)

In [19]:
bt = Backtest(GOOG, TrendFollow,
    # cash=10000, commission=.002,
    cash=1000,
    exclusive_orders=True, trade_on_close=True)

bt.run()

Backtest.run:   0%|          | 0/2118 [00:00<?, ?bar/s]

Start                     2004-08-19 00:00:00
End                       2013-03-01 00:00:00
Duration                   3116 days 00:00:00
Exposure Time [%]                     52.3743
Equity Final [$]                   2893.70796
Equity Peak [$]                    2895.68796
Return [%]                           189.3708
Buy & Hold Return [%]               522.06019
Return (Ann.) [%]                     13.2758
Volatility (Ann.) [%]                19.67027
CAGR [%]                              8.97307
Sharpe Ratio                          0.67492
Sortino Ratio                         1.22669
Calmar Ratio                          0.71282
Alpha [%]                           172.79392
Beta                                  0.03175
Max. Drawdown [%]                   -18.62436
Avg. Drawdown [%]                    -4.81521
Max. Drawdown Duration      445 days 00:00:00
Avg. Drawdown Duration       61 days 00:00:00
# Trades                                   47
Win Rate [%]                      

In [20]:
_ = bt.plot(filename="plot.html", open_browser=False)